In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Augmented (Secondly) Dataset.csv')

# Drop diseases with less than 1000 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 750].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 114
Number of rows left: 114312


In [3]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune for KNN
    n_neighbors = trial.suggest_int('n_neighbors', 3, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    p = trial.suggest_int('p', 1, 2)  # p=1 (Manhattan), p=2 (Euclidean)

    # Create KNeighborsClassifier with hyperparameters
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        p=p
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_neighbors={n_neighbors}, weights={weights}, p={p}, Accuracy={accuracy:.4f}")

    return accuracy

In [4]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize', 
    study_name="knn_diseases_symptoms_dropextremelymore750withoutSMOTE_SecondAugmentation_study",
    storage=r"sqlite:///C:/Users/khiew/Downloads/knn.db", 
    load_if_exists=True
)

# Optimize the study with your objective function, adjust n_trials as needed
study.optimize(objective, n_trials=20)

# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

# After finding the best hyperparameters, you can fit the final KNN model on resampled data
best_params = study.best_trial.params
final_model = KNeighborsClassifier(
    n_neighbors=best_params['n_neighbors'],
    weights=best_params['weights'],
    p=best_params['p']
)
final_model.fit(X_train, y_train)

# Evaluate on test set
y_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")


[I 2025-04-27 16:41:28,492] A new study created in RDB with name: knn_diseases_symptoms_dropextremelymore750withoutSMOTE_SecondAugmentation_study
[I 2025-04-27 16:52:22,618] Trial 0 finished with value: 0.6579842514544068 and parameters: {'n_neighbors': 29, 'weights': 'uniform', 'p': 1}. Best is trial 0 with value: 0.6579842514544068.


Trial 0: n_neighbors=29, weights=uniform, p=1, Accuracy=0.6580


[I 2025-04-27 16:54:19,175] Trial 1 finished with value: 0.6660214631249605 and parameters: {'n_neighbors': 25, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 0.6660214631249605.


Trial 1: n_neighbors=25, weights=distance, p=2, Accuracy=0.6660


[I 2025-04-27 16:55:58,498] Trial 2 finished with value: 0.658192005693414 and parameters: {'n_neighbors': 13, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.6660214631249605.


...

Trial 17: n_neighbors=42, weights=distance, p=1, Accuracy=0.6671


[I 2025-04-27 18:10:27,790] Trial 18 finished with value: 0.6680554158386667 and parameters: {'n_neighbors': 32, 'weights': 'distance', 'p': 1}. Best is trial 18 with value: 0.6680554158386667.


Trial 18: n_neighbors=32, weights=distance, p=1, Accuracy=0.6681


[I 2025-04-27 18:15:39,674] Trial 19 finished with value: 0.6674211721464569 and parameters: {'n_neighbors': 31, 'weights': 'distance', 'p': 1}. Best is trial 18 with value: 0.6680554158386667.


Trial 19: n_neighbors=31, weights=distance, p=1, Accuracy=0.6674

Best Trial:
FrozenTrial(number=18, state=TrialState.COMPLETE, values=[0.6680554158386667], datetime_start=datetime.datetime(2025, 4, 27, 18, 5, 23, 583506), datetime_complete=datetime.datetime(2025, 4, 27, 18, 10, 27, 767153), params={'n_neighbors': 32, 'weights': 'distance', 'p': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_neighbors': IntDistribution(high=50, log=False, low=3, step=1), 'weights': CategoricalDistribution(choices=('uniform', 'distance')), 'p': IntDistribution(high=2, log=False, low=1, step=1)}, trial_id=272, value=None)
Best Hyperparameters:
{'n_neighbors': 32, 'weights': 'distance', 'p': 1}
Test Accuracy: 0.6729
